# Ariel Quantum Training Notebook

This notebook runs the current Ariel five-gas hybrid regressor end to end. It supports classical stage-1 pretraining, hybrid stage-2 fine-tuning, or stage-2-only restart from an existing stage-1 checkpoint.

The notebook is wired to the code in `models/ariel_quantum_regression/`, so changes to the training code are reused here automatically.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from datetime import datetime
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "models" / "ariel_quantum_regression" / "training.py").exists():
            return candidate
        nested_root = candidate / "ariel" / "models" / "notebooks"
        if (nested_root / "models" / "ariel_quantum_regression" / "training.py").exists():
            return nested_root
    raise FileNotFoundError("Could not locate the Ariel notebook project root.")


def find_data_root(project_root: Path) -> Path:
    override = os.environ.get("ARIEL_DATA_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "TrainingData" / "AuxillaryTable.csv").exists() and (candidate / "TestData" / "AuxillaryTable.csv").exists():
            return candidate

    candidates = [
        project_root.parent.parent,
        project_root / "data" / "ariel-ml-dataset",
        project_root / "data" / "full-ariel",
    ]
    for candidate in candidates:
        if (candidate / "TrainingData" / "AuxillaryTable.csv").exists() and (candidate / "TestData" / "AuxillaryTable.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Ariel dataset root. Set ARIEL_DATA_ROOT if needed.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = find_data_root(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.style.use('seaborn-v0_8-whitegrid')
print(PROJECT_ROOT)
print(DATA_ROOT)
print({'cuda': torch.cuda.is_available(), 'mps': getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available()})


In [ ]:
# All hyperparameters and runtime knobs live in this cell.

RUN_MODE = 'two_stage'  # 'stage1_only' | 'stage2_only' | 'two_stage'
OUTPUT_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / f"ariel_quantum_notebook_{OUTPUT_TAG}"
CACHE_DIR = OUTPUT_ROOT / "prepared_cache"
EXISTING_STAGE1_CHECKPOINT = None  # Set this for RUN_MODE == "stage2_only"
STOP_AFTER_STAGE1 = False

COMMON_HP = {
    'seed': 42,
    'dropout': 0.05,
    'loss_name': 'mse',
    'weight_decay': 1.0e-4,
    'gradient_clip_norm': 5.0,
    'use_amp': True,
    'log_every_batches': 0,
    'log_every_epochs': 5,
    'qnn_qubits': 8,
    'qnn_depth': 2,
    'qnn_init_scale': 0.1,
    'quantum_device': 'lightning.gpu' if torch.cuda.is_available() else 'lightning.qubit',
    'quantum_use_async': False,
    'training_device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

STAGE1_HP = {
    'batch_size': 256,
    'eval_batch_size': 512,
    'max_epochs': 45,
    'early_stop_patience': 10,
    'scheduler_patience': 2,
    'scheduler_factor': 0.5,
    'classical_lr': 1.0e-3,
}

STAGE2_HP = {
    'batch_size': 16,
    'eval_batch_size': 32,
    'max_epochs': 30,
    'early_stop_patience': 8,
    'scheduler_patience': 3,
    'scheduler_factor': 0.5,
    'classical_lr': 5.0e-5,
    'quantum_lr': 2.0e-4,
    'quantum_warmup_epochs': 0,
    'quantum_ramp_epochs': 12,
    'quantum_backbone_freeze_epochs': 6,
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_MODE =', RUN_MODE)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('COMMON_HP =')
print(json.dumps(COMMON_HP, indent=2))
print('STAGE1_HP =')
print(json.dumps(STAGE1_HP, indent=2))
print('STAGE2_HP =')
print(json.dumps(STAGE2_HP, indent=2))


In [ ]:
import importlib
import models.ariel_quantum_regression.model as exobiome_model_module
import models.ariel_quantum_regression.training as exobiome_training_module
importlib.reload(exobiome_model_module)
importlib.reload(exobiome_training_module)
TrainingConfig = exobiome_training_module.TrainingConfig
run_training_experiment = exobiome_training_module.run_training_experiment


In [ ]:
def make_stage1_config(output_dir: Path) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(CACHE_DIR),
        seed=COMMON_HP["seed"],
        batch_size=STAGE1_HP["batch_size"],
        eval_batch_size=STAGE1_HP["eval_batch_size"],
        max_epochs=STAGE1_HP["max_epochs"],
        early_stop_patience=STAGE1_HP["early_stop_patience"],
        scheduler_patience=STAGE1_HP["scheduler_patience"],
        scheduler_factor=STAGE1_HP["scheduler_factor"],
        classical_lr=STAGE1_HP["classical_lr"],
        weight_decay=COMMON_HP["weight_decay"],
        gradient_clip_norm=COMMON_HP["gradient_clip_norm"],
        dropout=COMMON_HP["dropout"],
        loss_name=COMMON_HP["loss_name"],
        qnn_qubits=COMMON_HP["qnn_qubits"],
        qnn_depth=COMMON_HP["qnn_depth"],
        qnn_init_scale=COMMON_HP["qnn_init_scale"],
        quantum_device=COMMON_HP["quantum_device"],
        quantum_use_async=COMMON_HP["quantum_use_async"],
        training_device=COMMON_HP["training_device"],
        classical_only=True,
        use_amp=COMMON_HP["use_amp"],
        log_every_batches=COMMON_HP["log_every_batches"],
        log_every_epochs=COMMON_HP["log_every_epochs"],
    )

def make_stage2_config(output_dir: Path, checkpoint_path: Path) -> TrainingConfig:
    return TrainingConfig(
        project_root=str(PROJECT_ROOT),
        data_root=str(DATA_ROOT),
        output_dir=str(output_dir),
        prepared_cache_dir=str(CACHE_DIR),
        init_checkpoint_path=str(checkpoint_path),
        seed=COMMON_HP["seed"],
        batch_size=STAGE2_HP["batch_size"],
        eval_batch_size=STAGE2_HP["eval_batch_size"],
        max_epochs=STAGE2_HP["max_epochs"],
        early_stop_patience=STAGE2_HP["early_stop_patience"],
        scheduler_patience=STAGE2_HP["scheduler_patience"],
        scheduler_factor=STAGE2_HP["scheduler_factor"],
        classical_lr=STAGE2_HP["classical_lr"],
        quantum_lr=STAGE2_HP["quantum_lr"],
        weight_decay=COMMON_HP["weight_decay"],
        gradient_clip_norm=COMMON_HP["gradient_clip_norm"],
        dropout=COMMON_HP["dropout"],
        loss_name=COMMON_HP["loss_name"],
        qnn_qubits=COMMON_HP["qnn_qubits"],
        qnn_depth=COMMON_HP["qnn_depth"],
        qnn_init_scale=COMMON_HP["qnn_init_scale"],
        quantum_device=COMMON_HP["quantum_device"],
        quantum_use_async=COMMON_HP["quantum_use_async"],
        training_device=COMMON_HP["training_device"],
        quantum_warmup_epochs=STAGE2_HP["quantum_warmup_epochs"],
        quantum_ramp_epochs=STAGE2_HP["quantum_ramp_epochs"],
        quantum_backbone_freeze_epochs=STAGE2_HP["quantum_backbone_freeze_epochs"],
        classical_only=False,
        use_amp=COMMON_HP["use_amp"],
        log_every_batches=COMMON_HP["log_every_batches"],
        log_every_epochs=COMMON_HP["log_every_epochs"],
    )

def load_history(output_dir: Path) -> pd.DataFrame | None:
    history_path = output_dir / "history.csv"
    if not history_path.exists():
        return None
    return pd.read_csv(history_path)

def plot_history(output_dir: Path, title: str) -> None:
    history = load_history(output_dir)
    if history is None or history.empty:
        print(f"No history found for {title}: {output_dir}")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["epoch"], history["train_rmse_mean"], label="train_rmse_mean")
    ax.plot(history["epoch"], history["val_rmse_mean"], label="val_rmse_mean")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel("RMSE")
    ax.legend()
    plt.show()

def print_summary(output_dir: Path) -> None:
    summary_path = output_dir / "run_summary.json"
    if summary_path.exists():
        print(summary_path)
        print(summary_path.read_text())
    else:
        print(f"No run_summary.json found for {output_dir}")


## Stage 1

Run this cell to train the classical backbone unless you are doing a stage-2-only restart from an existing checkpoint.

In [ ]:
stage1_output_dir = OUTPUT_ROOT / "stage1_classical"
stage1_result = None

if RUN_MODE in {"stage1_only", "two_stage"}:
    stage1_cfg = make_stage1_config(stage1_output_dir)
    print(json.dumps(stage1_cfg.to_json_dict(), indent=2))
    stage1_result = run_training_experiment(stage1_cfg)
    print(json.dumps(stage1_result["summary"], indent=2))
else:
    print("Stage 1 skipped because RUN_MODE is stage2_only.")


In [ ]:
if RUN_MODE == "stage2_only":
    if EXISTING_STAGE1_CHECKPOINT is None:
        raise ValueError("Set EXISTING_STAGE1_CHECKPOINT when RUN_MODE is stage2_only.")
    stage1_checkpoint_path = Path(EXISTING_STAGE1_CHECKPOINT).expanduser().resolve()
else:
    stage1_checkpoint_path = (stage1_output_dir / "best_model.pt").resolve()

print(stage1_checkpoint_path)
print(stage1_checkpoint_path.exists())


## Stage 2

Run this cell to fine-tune the hybrid model from the best stage-1 checkpoint.

In [ ]:
stage2_output_dir = OUTPUT_ROOT / "stage2_hybrid"
stage2_result = None

if RUN_MODE in {"stage2_only", "two_stage"} and not STOP_AFTER_STAGE1:
    stage2_cfg = make_stage2_config(stage2_output_dir, stage1_checkpoint_path)
    print(json.dumps(stage2_cfg.to_json_dict(), indent=2))
    stage2_result = run_training_experiment(stage2_cfg)
    print(json.dumps(stage2_result["summary"], indent=2))
else:
    print("Stage 2 skipped.")


## Inspect Results

In [ ]:
for label, output_dir in [("Stage 1", stage1_output_dir), ("Stage 2", stage2_output_dir)]:
    print(f"\n=== {label}: {output_dir} ===")
    print_summary(output_dir)
    plot_history(output_dir, label)


In [ ]:
artifact_candidates = [
    stage1_output_dir / "best_model.pt",
    stage2_output_dir / "best_model.pt",
    stage1_output_dir / "testdata_predictions.csv",
    stage2_output_dir / "testdata_predictions.csv",
]

for candidate in artifact_candidates:
    print(candidate, candidate.exists())
